# Read me

This notebook uses a NearestNeighbors model with Euclidean distance to recommend 20 tracks based on the profile of 10 input tracks. The profile is built by calculating the mean of audio features of the input tracks. 

9 features → k-NN → 20 recommandations

In [23]:
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

import joblib

import app.spotify.spotify as spo


# Load dataset

In [24]:
df = pd.read_csv("../raw_data/spotify-tracks-dataset-detailed.csv")
print(f'Loaded {len(df)} rows')

Loaded 114000 rows


# Cleaning dataset
## Removing Comedy genre

In [25]:
len_before = len(df)
df = df[df["track_genre"] != "comedy"].reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} comedy tracks. Left {len_after} rows')

Removed 1000 comedy tracks. Left 113000 rows


## Drop duplicates

In [26]:
len_before = len(df)
df = df.drop_duplicates(subset=["track_id"]).drop_duplicates(subset=["artists", "track_name"]).reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} duplicates. Left {len_after} rows')

Removed 32642 duplicates. Left 80358 rows


## Drop missing values

In [27]:
len_before = len(df)
df = df.dropna(subset=['album_name']).dropna(subset=['track_name']).reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} null values. Left {len(df)} rows')

Removed 1 null values. Left 80357 rows


In [28]:
# save catalog
df.to_parquet("../artifacts/catalog_clean.parquet", index=False)

## Scaling

In [18]:
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

scaler = StandardScaler().set_output(transform="pandas")

X = df[audio_features]

X_scaled = scaler.fit_transform(X)

In [21]:
# save scaler
joblib.dump(scaler, "../artifacts/baseline/baseline_scaler.joblib")

['../artifacts/baseline/baseline_scaler.joblib']

In [ ]:
# save scaled catalog
X_scaled.to_parquet("../artifacts/baseline/catalog_scaled.parquet", index=False)

In [12]:
X_scaled.head()

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.655383,-0.667949,0.343899,0.742394,-0.865010,-0.561572,0.753029,0.953320,-1.147548
1,-0.779302,-1.805605,-1.632471,-0.052910,1.771528,-0.561558,-0.593292,-0.744136,-1.494515
2,-0.678426,-1.061308,-0.219109,-0.298536,-0.339358,-0.561575,-0.509475,-1.301114,-1.533011
3,-1.642355,-2.215933,-1.873653,-0.529854,1.715356,-0.561362,-0.430896,-1.213967,1.974189
4,0.330337,-0.737365,-0.209123,-0.335499,0.426356,-0.561575,-0.688111,-1.123032,-0.081759


In [13]:
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


# Model 1
- NearestNeighbors with Euclidean distance
- Input tracks: Average acoustic profile (Centroid k-NN)

## Train

In [9]:
# NearestNeighbors with Euclidean distance
model = NearestNeighbors(metric="euclidean")

model.fit(X_scaled)

NearestNeighbors(metric='euclidean')

In [ ]:
# Save model

joblib.dump(model, "../artifacts/baseline/baseline_knn_model.joblib")

## Input tracks

In [15]:
# Build an input track set of 10 tracks
input_tracks = X_scaled.sample(n=10)

In [16]:
# Print input track names
input_indices = input_tracks.index.to_list()
df.iloc[input_indices][['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]

,track_id,artists,album_name,track_name,track_genre
10847,1L4ntmBIXfdF6mZSbWhT5D,Fred Cheng,"逆著風 (劇集 ""失憶24小時"" 主題曲)","逆著風 - 劇集 ""大步走"" 主題曲",cantopop
47062,0OKaPDf7ljRTHyEWfIEICn,Perfume,"Perfume The Best ""P Cubed""",リニアモーターガール,j-idol
44613,2bHRnMtpXMUmzDHZrAyxne,Rammstein,Rammstein,Hallomann,industrial
16708,3FskQrDXcY24ur2fCvz35O,Burna Boy,Outside,Ye,dancehall
20226,5CGziaEatzKez145qf7Xj2,Irene Cara,Nyårsmusik,Flashdance... What A Feeling - '95 Version,disco
69893,3VyXwM2xemrXttbWUb004d,Kerry Butler;Curtis Holbrook;Anika Larsen;Keni...,Xanadu: Original Broadway Cast Recording,I'm Alive,show-tunes
18205,7q8TrL8QgMP9lWTcaYmGZ5,D.F.C.,Igreja Quadrangular Do Triângulo Redondo,Lucro é o Fim,death-metal
62697,1JiKG47wsKUFSJwzmSkMwN,Attaque 77,Karmagedon,Cartonero,punk-rock
49791,6wMJyFGW2gzVkYqOkYSmPs,Pj Panda;Leo Lion,Pj Panda's BedTime Beats,Leo Lullaby,kids
14714,1DOgO7ek31amB3SF2g1FVH,Infraction,Almost Evil,Almost Evil,club


In [4]:
token = spo.get_access_token()
playlist_id = spo.create_playlist(token)
spo.add_tracks_to_playlist(token, playlist_id, df.iloc[input_indices]['track_id'])
spo.display_player(playlist_id)

NameError: name 'spo' is not defined

In [18]:
# Calculate input profile
input_profile = input_tracks.mean(axis=0).to_frame().T

input_profile

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,-0.137617,0.336274,0.548828,0.071691,-0.564382,-0.045335,-0.290396,0.089815,0.790622


## Recommend

In [19]:
# Find 20 neighbors
distances, indices = model.kneighbors(input_profile, n_neighbors=20)

# The model returns an array of indices
# Return the tracks from the original dataframe
recommendations = df.iloc[indices[0]]


In [20]:
# Print recommendations profile
recommendations_indices = recommendations.index.to_list()
recommendations_tracks = X_scaled.iloc[recommendations_indices]

recommendations_profile = recommendations_tracks.mean(axis=0).to_frame().T

recommendations_profile

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,-0.087739,0.341673,0.504991,-0.18085,-0.582112,-0.500308,-0.411382,0.028623,0.77791


In [21]:
playlist_id = spo.create_playlist(token)
spo.add_tracks_to_playlist(token, playlist_id, df.iloc[recommendations_indices]['track_id'])
spo.display_player(playlist_id)

In [22]:
recommendations[['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]

,track_id,artists,album_name,track_name,track_genre
25048,6Dg3ArDiJcseYoeeTZZXK6,San Holo,All The Highs,All The Highs,electronic
51830,73pv8Je9LeooK0UgE9mXRY,Havoc Mathan;Havoc Naven,Best of Havoc Brothers,Pirantha Naal,malay
37078,7axJf0rBRxyTk397LbJpTt,Pitty;Drik Barbosa;WEKS,Casulo,Diamante,hard-rock
50606,3CpD9lnEHelRkBtsdktwHy,Luis Fonsi,Ya huele a buñuelos,Let It Snow,latin
60440,3fVnlF4pGqWI9flVENcT28,Taylor Swift,1989,Wildest Dreams,pop
31329,2rcELfYmKTFtit6Gv1ZEqF,Canção & Louvor,Eu Sou Mistério,Eu Sou Mistério,gospel
66705,7jahcsevTvePvdduUFVzpq,Aries,WELCOME HOME,CAROUSEL,sad
39276,5oYRR7SsbqnpXKaXAwbH5m,A.N.I.M.A.L.;Chuck Jonhson,6 (Animal),Buscando llegar hasta el sol,heavy-metal
31439,1wlAJE1E8Dom0LwgAImVg9,Lauriete,João Viu,João Viu,gospel
51769,4QzJOTh94tmzsR1lV6mfML,Sajeer Koppam,"Dilse, Vol. 2",Vannu ninte,malay


# Alternative models
- Model: NearestNeighbors with Manhattan distance
- Input tracks: average of multimodal preferences (Multi-seed k-NN)

['../app/models/baseline_knn_model.joblib']